In [1]:
import glob
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rio
from geocube.api.core import make_geocube
from IPython.display import Image
from matplotlib.colors import ListedColormap

# Vary NDVI threshold

See e-mail from Mathias received 02.06.2025.

## 1. Datasets to process

In [2]:
# Read list of datasets to process
ds_list_xlsx = r"../data/halden_baselevels_and_paths.xlsx"
df = pd.read_excel(ds_list_xlsx)
df = df.query("to_be_analysed == 'y'").reset_index(drop=True)
df.head()

,site,base_line_height,base_height_coordinates,stream_id,stream_width,to_be_analysed,dsm,ndvi
0,H20 L,192.066299,"11°26'1,229""E 59°56'26,902""N",H20a,1.5,y,niva-tidy/2023/niva_202308301149_halden_h20_rg...,niva-tidy/2023/niva_202308301136_halden_h20_ms...
1,H20 R,192.705368,"11°26'1,808""E 59°56'26,932""N",H20b,1.5,y,niva-tidy/2023/niva_202308301149_halden_h20_rg...,niva-tidy/2023/niva_202308301136_halden_h20_ms...
2,H21 L,192.179993,"11°26'52,241""E 59°56'27,869""N",H21a,2.5,y,niva-tidy/2023/niva_202308301109_halden_h21_rg...,niva-tidy/2023/niva_202308301118_halden_h21_ms...
3,H21 R,192.670471,"11°26'53,417""E 59°56'27,724""N",H21b,2.5,y,niva-tidy/2023/niva_202308301109_halden_h21_rg...,niva-tidy/2023/niva_202308301118_halden_h21_ms...
4,H22 L,161.835281,"11°34'44,749""E 59°53'1,058""N",H22a,2.5,y,niva-tidy/2023/niva_202308300741_halden_h22_rg...,niva-tidy/2023/niva_202308300752_halden_h22_ms...


## 2. Read vector data

In [3]:
# Paths to shapefiles from Mathias
str_shp_path = r"../gis/shapefiles/Streams.shp"
rip_shp_path = r"../gis/shapefiles/Riparian_buffers.shp"
veg_shp_path = r"../gis/shapefiles/Riparian_veg_width.shp"

# Read geodataframes
str_gdf = gpd.read_file(str_shp_path)
rip_gdf = gpd.read_file(rip_shp_path)
veg_gdf = gpd.read_file(veg_shp_path)

## 3. Vary shading

In [4]:
ndvi_thresh_list = sorted(list(np.arange(0.2, 0.625, 0.025)) + [0.26, 0.33])
xl_path = r"../data/vary_ndvi_thresh.xlsx"

In [5]:
# Dict to store results
res_list = []

for ndvi_thresh in ndvi_thresh_list:
    # Pixels within the buffer with values < 'ndvi_thresh' are considered to be unshaded;
    # pixels with values >= 'ndvi_thresh' are considered shaded.
    shade_dict = {
        "site": [],
        "shading_pct": [],
    }
    for idx, row in df.iterrows():
        width = row["stream_width"]
        str_id = row["stream_id"]
        site = row["site"]
        site_main = site[:3]

        # Get vector data for site
        site_str_gdf = str_gdf.query("Stream_id == @site_main").copy()
        site_rip_gdf = rip_gdf.query("Stream_id == @str_id").copy()


        # Buffer the stream
        site_str_gdf["geometry"] = site_str_gdf.buffer(width / 2)

        # Clip stream to riparian poly
        site_str_gdf = gpd.clip(site_str_gdf, site_rip_gdf, keep_geom_type=True)

        # Read NDVI data
        ndvi_tif_path = os.path.join(
            "/home/notebook/shared-seabee-ns9879k", row["ndvi"]
        )
        ndvi_data = rio.open_rasterio(ndvi_tif_path, mask_and_scale=True)

        # Reproject if necessary
        site_str_gdf = site_str_gdf.to_crs(ndvi_data.rio.crs)

        # Extract pixels within buffer
        masked_ndvi = ndvi_data.rio.clip(site_str_gdf.geometry, site_str_gdf.crs)

        # Calculate the proportion of pixels with values >= ndvi_thresh
        buffer_pixels = masked_ndvi.notnull().sum().item()
        shaded_pixels = (masked_ndvi >= ndvi_thresh).sum().item()
        shade_pct = 100 * shaded_pixels / buffer_pixels

        # Results to dict
        shade_dict["site"].append(site)
        shade_dict["shading_pct"].append(shade_pct)

    # Build results dataframe
    shade_df = pd.DataFrame(shade_dict)
    shade_df[f"shading_pct_{np.round(ndvi_thresh,3):.3f}"] = shade_df["shading_pct"].round(1)
    del shade_df["shading_pct"]
    shade_df = shade_df.set_index("site")
    res_list.append(shade_df)

res_df = pd.concat(res_list, axis="columns").reset_index()
res_df.to_excel(xl_path, index=False)
res_df.head()

,site,shading_pct_0.200,shading_pct_0.225,shading_pct_0.250,shading_pct_0.260,shading_pct_0.275,shading_pct_0.300,shading_pct_0.325,shading_pct_0.330,shading_pct_0.350,shading_pct_0.375,shading_pct_0.400,shading_pct_0.425,shading_pct_0.450,shading_pct_0.475,shading_pct_0.500,shading_pct_0.525,shading_pct_0.550,shading_pct_0.575,shading_pct_0.600
0,H20 L,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,H20 R,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2,H21 L,99.2,98.8,98.4,98.3,98.1,97.7,97.2,97.1,96.6,96.0,95.2,94.2,93.0,91.5,90.0,87.7,85.3,82.3,78.8
3,H21 R,99.3,99.1,98.9,98.8,98.7,98.4,98.0,98.0,97.6,97.1,96.5,95.7,94.6,93.5,92.3,90.4,88.4,86.1,83.6
4,H22 L,44.8,43.2,42.1,41.3,40.8,39.9,38.9,38.9,38.1,37.3,36.6,35.8,35.0,34.1,33.5,32.6,31.9,31.0,30.2
